<a href="https://colab.research.google.com/github/dejanjovic1283-ui/imdb-bert-sentiment-analysis/blob/main/imdb_bert_sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1: Install Required Libraries

In this step, we install the required libraries for running BERT-based sentiment analysis with Hugging Face Transformers and PyTorch.

In [1]:
!pip install -q transformers torch accelerate

# Step 2: Import Libraries

We import the Hugging Face pipeline API to perform sentiment analysis inference with a pretrained transformer model.

In [2]:
from transformers import pipeline

# Step 3: Load a Pretrained Sentiment Analysis Pipeline

We load a pretrained sentiment analysis model using the Hugging Face pipeline API.

In [3]:
classifier = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

# Step 4: Test the Model on Example Reviews

We test the pretrained BERT-based sentiment classifier on a few movie review examples.

In [4]:
reviews = [
    "This movie was absolutely amazing. I loved every minute of it.",
    "This was one of the worst films I have ever seen.",
    "The acting was decent, but the story felt weak and predictable.",
    "A brilliant film with powerful performances and excellent direction."
]

for review in reviews:
    result = classifier(review)
    print("Review:", review)
    print("Prediction:", result[0]["label"])
    print("Score:", result[0]["score"])
    print("-" * 80)

Review: This movie was absolutely amazing. I loved every minute of it.
Prediction: POSITIVE
Score: 0.9998840093612671
--------------------------------------------------------------------------------
Review: This was one of the worst films I have ever seen.
Prediction: NEGATIVE
Score: 0.9997578263282776
--------------------------------------------------------------------------------
Review: The acting was decent, but the story felt weak and predictable.
Prediction: NEGATIVE
Score: 0.9994146823883057
--------------------------------------------------------------------------------
Review: A brilliant film with powerful performances and excellent direction.
Prediction: POSITIVE
Score: 0.9998795986175537
--------------------------------------------------------------------------------


# Step 5: Create a Simple Custom Prediction Function

We create a helper function so that we can easily test any custom movie review.

In [5]:
def predict_review_sentiment(text):
    result = classifier(text)[0]
    return {
        "review": text,
        "prediction": result["label"],
        "score": round(result["score"] * 100, 2)
    }

# Step 6: Test a Custom Review

We test the function on a single custom review input.

In [6]:
custom_review = "This movie had an emotional story and fantastic acting."
prediction = predict_review_sentiment(custom_review)
prediction

{'review': 'This movie had an emotional story and fantastic acting.',
 'prediction': 'POSITIVE',
 'score': 99.99}

# Step 7: Save Results for Documentation

We save a few sample predictions that can later be added to the README file or GitHub project description.

In [7]:
sample_reviews = [
    "This movie was absolutely amazing.",
    "I did not enjoy this movie at all.",
    "The visuals were beautiful, but the plot was weak."
]

for review in sample_reviews:
    print(predict_review_sentiment(review))

{'review': 'This movie was absolutely amazing.', 'prediction': 'POSITIVE', 'score': 99.99}
{'review': 'I did not enjoy this movie at all.', 'prediction': 'NEGATIVE', 'score': 99.9}
{'review': 'The visuals were beautiful, but the plot was weak.', 'prediction': 'NEGATIVE', 'score': 99.89}


# Step 8: Install Additional Libraries for Fine-Tuning

We install the datasets and evaluate libraries required for dataset handling and model evaluation.

In [8]:
!pip install -q datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.8 MB/s eta 0:00:00


# Step 9: Import Training Libraries

We import the tokenizer, model, dataset utilities, and Trainer components required for BERT fine-tuning.

In [9]:
import numpy as np
from datasets import load_dataset
import evaluate

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

# Step 10: Load the IMDB Dataset

We load the IMDB dataset for binary sentiment classification.

In [10]:
dataset = load_dataset("imdb")
dataset

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

# Step 11: Load the Tokenizer

We use a pretrained DistilBERT tokenizer for text preprocessing.

In [11]:
checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Step 12: Tokenize the Dataset

We tokenize the text data so it can be used by the transformer model.

In [12]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

# Step 13: Prepare a Smaller Training Subset

To keep training faster in Colab, we create smaller train and test subsets.

In [13]:
small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(2000))
small_test_dataset = tokenized_dataset["test"].shuffle(seed=42).select(range(1000))

# Step 14: Load the Sequence Classification Model

We load a pretrained transformer model for binary sentiment classification.

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Step 15: Prepare Evaluation Metrics

We use accuracy as the main evaluation metric for this project.

In [15]:
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

# Step 16: Create the Data Collator

The data collator dynamically pads input sequences during training.

In [16]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Step 17: Define Training Arguments

We define the main training configuration for fine-tuning the transformer model.

In [17]:
training_args = TrainingArguments(
    output_dir="imdb_bert_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_dir="logs",
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


# Step 18: Initialize the Trainer

We create a Trainer instance to manage the training and evaluation workflow.

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_test_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Step 19: Fine-Tune the Model

We fine-tune the pretrained DistilBERT model on the IMDB sentiment dataset.

In [20]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.324612,0.872000
2,0.308376,0.387610,0.888000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=0.30837567138671873, metrics={'train_runtime': 236.2716, 'train_samples_per_second': 16.93, 'train_steps_per_second': 2.116, 'total_flos': 490344634550016.0, 'train_loss': 0.30837567138671873, 'epoch': 2.0})

# Step 20: Evaluate the Fine-Tuned Model

We evaluate the model on the test subset.

In [21]:
trainer.evaluate()

{'eval_loss': 0.3876095712184906,
 'eval_accuracy': 0.888,
 'eval_runtime': 14.6533,
 'eval_samples_per_second': 68.244,
 'eval_steps_per_second': 8.53,
 'epoch': 2.0}

# Step 21: Save the Fine-Tuned Model

We save the fine-tuned transformer model and tokenizer for later inference or deployment.

In [22]:
model.save_pretrained("imdb_distilbert_model")
tokenizer.save_pretrained("imdb_distilbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('imdb_distilbert_model/tokenizer_config.json',
 'imdb_distilbert_model/tokenizer.json')

# Step 22: Run Inference with the Fine-Tuned Model

We create a Hugging Face pipeline using the fine-tuned model.

In [23]:
from transformers import pipeline

fine_tuned_classifier = pipeline(
    "sentiment-analysis",
    model="imdb_distilbert_model",
    tokenizer="imdb_distilbert_model"
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

# Step 23: Test the Fine-Tuned Model

We test the saved model on new review examples.

In [25]:
label_map = {
    "LABEL_0": "Negative",
    "LABEL_1": "Positive"
}

for review in test_reviews:
    result = fine_tuned_classifier(review)
    raw_label = result[0]["label"]
    mapped_label = label_map.get(raw_label, raw_label)

    print("Review:", review)
    print("Prediction:", mapped_label)
    print("Score:", result[0]["score"])
    print("-" * 80)

Review: This movie was fantastic and very emotional.
Prediction: Positive
Score: 0.9899179339408875
--------------------------------------------------------------------------------
Review: I hated every second of this film.
Prediction: Negative
Score: 0.9409971237182617
--------------------------------------------------------------------------------
